# PANALDO | Interactive normal-mode viewer v3
Rebuilds the approved single-page v3 HTML layout from the original ORCA Hessian files. Place `PANALDO_PHASE_I.tar.gz` beside the notebook, or extract the Hessian files into `01_PROTONATION/`. Run all cells. The output retains the four molecular species and 225 positive modes. The original approved HTML is separately retained in the repository. Bond-motion rankings are first-order geometric derivatives, not formal vibrational assignments.


In [ ]:
from pathlib import Path
import tarfile, re, json
import numpy as np
import pandas as pd
from IPython.display import display, HTML, IFrame
names = ['P01_neutral','P02_carbonyl_O','P03_amide_N','P04_phenolic_O']
archive = Path('PANALDO_PHASE_I.tar.gz')
def get_hess(name):
    local = Path('01_PROTONATION') / (name+'.hess')
    if local.exists(): return local.read_text()
    with tarfile.open(archive) as t:
        matches = [m for m in t.getmembers() if Path(m.name).name == name+'.hess']
        assert len(matches)==1
        return t.extractfile(matches[0]).read().decode()
def section(s, key):
    return [x.strip() for x in re.search(r'^\$'+key+r'\s*\n(.*?)(?=^\$|\Z)',s,re.M|re.S).group(1).splitlines() if x.strip() and not x.strip().startswith("#")]
def parse(name):
    s=get_hess(name); a=section(s,'atoms'); N=int(a[0]); rows=[x.split() for x in a[1:]]
    assert len(rows)==N
    symbols=[r[0] for r in rows]; xyz=np.array([[float(v) for v in r[2:5]] for r in rows])*0.529177210903
    f=section(s,'vibrational_frequencies'); freq=np.array([float(x.split()[1]) for x in f[1:]])
    ir=section(s,'ir_spectrum'); intens=np.array([float(x.split()[2]) for x in ir[1:]])
    # ORCA IR spectrum columns: frequency, epsilon, intensity, TX, TY, TZ.
    m=section(s,'normal_modes'); nr,nc=map(int,m[0].split()); M=np.full((nr,nc),np.nan); i=1
    while i<len(m):
        cols=list(map(int,m[i].split())); i+=1
        for j in range(nr):
            r=m[i].split(); i+=1; M[int(r[0]),cols]=list(map(float,r[1:]))
    assert nr==nc==3*N==len(freq)==len(intens) and np.isfinite(M).all()
    assert np.allclose(freq, [float(x.split()[0]) for x in ir[1:]],atol=.011)
    radii={'C':.76,'N':.71,'O':.66,'H':.31}
    bonds=[[i,j] for i in range(N) for j in range(i+1,N) if np.linalg.norm(xyz[i]-xyz[j]) < 1.25*(radii[symbols[i]]+radii[symbols[j]])]
    modes=[]
    for k in np.where(freq>0)[0]:
        d=M[:,k].reshape(N,3); d=d/np.linalg.norm(d,axis=1).max()
        modes.append(dict(mode=int(k),frequency=float(freq[k]),intensity=float(intens[k]),d=d.tolist()))
    return dict(symbols=symbols,xyz=xyz.tolist(),bonds=bonds,modes=modes)
data={name:parse(name) for name in names}
assert [len(data[n]['modes']) for n in names]==[54,57,57,57]
summary=pd.DataFrame([{'Species':n,'Atoms':len(v['symbols']),'Positive modes':len(v['modes']),'Lowest [cm⁻¹]':min(m['frequency'] for m in v['modes'])} for n,v in data.items()]); display(summary)


## Mode assignment guide
Choose a mode and examine the changing C–O, C–N, O–H and N–H distances. The largest displacement alone is not an assignment. Coupled modes can involve several bonds; the bond-change table ranks the absolute first-order bond-length derivative at the equilibrium structure, not a potential-energy distribution. Use the sign and magnitude to guide visual interpretation.

In [ ]:
targets=[48,52,52,52]
rows=[]
for name,k in zip(names,targets):
    v=data[name]; m=next(x for x in v['modes'] if x['mode']==k); x=np.array(v['xyz']); d=np.array(m['d'])
    for i,j in v['bonds']:
        r=x[j]-x[i]; change=np.dot(r/np.linalg.norm(r),d[j]-d[i]); rows.append(dict(species=name,mode=k,frequency=m['frequency'],bond=f"{v['symbols'][i]}{i+1}–{v['symbols'][j]}{j+1}",signed_derivative=change,absolute_derivative=abs(change)))
assignments=pd.DataFrame(rows).sort_values(['species','absolute_derivative'],ascending=[True,False]); display(assignments.groupby('species',sort=False).head(5).reset_index(drop=True)); assignments.to_csv('PANALDO_mode_bond_changes.csv',index=False)


In [ ]:
# Rebuild approved v3 viewer with parsed ORCA data
labels = {"P01_neutral":"Neutral paracetamol","P02_carbonyl_O":"Carbonyl-O-protonated paracetamol","P03_amide_N":"Amide-N-protonated paracetamol","P04_phenolic_O":"Phenolic-O-protonated paracetamol"}
defaults = {"P01_neutral":48,"P02_carbonyl_O":52,"P03_amide_N":52,"P04_phenolic_O":52}
viewer_data = {}
for name, v in data.items():
    x = np.array(v['xyz'])
    x = x - x.mean(axis=0)
    modes = []
    for m in v['modes']:
        d = np.array(m['d'])
        ranking = []
        for i,j in v['bonds']:
            r = x[j] - x[i]
            change = float(np.dot(r / np.linalg.norm(r), d[j] - d[i]))
            ranking.append(dict(bond=f"{v['symbols'][i]}{i+1}–{v['symbols'][j]}{j+1}", value=change, abs=abs(change), atoms=[i,j]))
        ranking.sort(key=lambda a: a['abs'], reverse=True)
        modes.append(dict(m, top_bonds=ranking[:5]))
    viewer_data[name] = dict(label=labels[name], symbols=v['symbols'], xyz=x.tolist(), bonds=v['bonds'], modes=modes, default_mode=defaults[name])
assert sum(len(v['modes']) for v in viewer_data.values()) == 225
template = "<!doctype html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">\n<title>PANALDO Normal Mode Viewer — all protonation states</title>\n<style>\n:root {\n  --bg:#f4f7fb;\n  --card:#ffffff;\n  --ink:#17212b;\n  --muted:#64748b;\n  --blue:#1d4ed8;\n  --blue2:#eff6ff;\n  --line:#e2e8f0;\n}\n*{box-sizing:border-box}\nbody{margin:0;font-family:\"Segoe UI\",Arial,sans-serif;color:var(--ink);background:\nradial-gradient(circle at top left,#e8f1ff 0%,transparent 28%),\nradial-gradient(circle at bottom right,#f2ecff 0%,transparent 28%),\nvar(--bg);}\n.wrap{max-width:1280px;margin:18px auto;padding:0 14px 18px;}\n.grid{display:grid;grid-template-columns:minmax(340px,1fr) 380px;gap:16px;align-items:start;}\n.card{background:var(--card);border:1px solid var(--line);border-radius:18px;box-shadow:0 10px 25px rgba(15,23,42,.08);overflow:hidden}\n.head{padding:16px 18px 10px;border-bottom:1px solid #eef2f7}\nh1{margin:0 0 6px;font-size:23px;font-weight:800;line-height:1.2}\n.sub{color:var(--muted);font-size:13px;line-height:1.5}\n.toolbar{display:grid;grid-template-columns:1fr 1fr;gap:10px;padding:14px 18px 8px;}\n.toolbar label{display:grid;gap:5px;font-size:12px;font-weight:800;color:#334155;text-transform:uppercase;letter-spacing:.03em}\nselect,button,input[type=range]{width:100%}\nselect{border:1px solid #dbe3ee;border-radius:12px;padding:10px 12px;background:#fff;font-size:14px;font-weight:600;color:#1f2937}\n.modebar{display:flex;flex-wrap:wrap;gap:8px;padding:0 18px 8px}\n.pill{padding:7px 10px;border-radius:999px;background:var(--blue2);color:#1e40af;font-size:12px;font-weight:800}\n.canvasbox{padding:10px 12px 10px}\ncanvas{width:100%;height:620px;display:block;border-radius:16px;border:1px solid #e8edf5;background:\nradial-gradient(circle at 35% 25%, #ffffff 0%, #f9fbff 40%, #eef4fb 100%);cursor:grab}\ncanvas:active{cursor:grabbing}\n.controls{padding:14px 16px 18px;border-top:1px solid #eef2f7;display:grid;gap:12px}\n.row{display:flex;align-items:center;gap:10px;flex-wrap:wrap}\nbutton{border:none;border-radius:12px;padding:10px 14px;background:var(--blue);color:white;font-weight:800;cursor:pointer;box-shadow:0 5px 14px rgba(29,78,216,.22)}\nbutton.secondary{background:#f3f4f6;color:#374151;box-shadow:none}\n.range-row{display:grid;grid-template-columns:88px 1fr auto;gap:10px;align-items:center}\n.range-row label{font-size:13px;font-weight:800;color:#334155}\n.small{color:var(--muted);font-size:12px;font-weight:700}\n.side{padding:16px;display:grid;gap:14px}\n.statbox{background:#f8fafc;border:1px solid #eef2f7;border-radius:14px;padding:12px 14px}\n.statgrid{display:grid;grid-template-columns:1fr 1fr;gap:10px 12px}\n.k{font-size:11px;color:var(--muted);text-transform:uppercase;letter-spacing:.04em;font-weight:800}\n.v{font-size:16px;font-weight:800;margin-top:3px}\n.legend{display:grid;grid-template-columns:1fr 1fr;gap:8px 12px}\n.leg{display:flex;align-items:center;gap:8px;font-size:13px;font-weight:700}\n.dot{width:12px;height:12px;border-radius:50%;box-shadow:inset 0 1px 2px rgba(255,255,255,.7),0 0 0 1px rgba(0,0,0,.08)}\n.note{line-height:1.55;font-size:13px;color:#334155}\n.list{margin:0;padding-left:18px}\n.list li{margin:0 0 6px;font-size:13px;line-height:1.45;color:#334155}\n.footer{color:var(--muted);font-size:12px}\n.hl{background:#fff8c5;border:1px solid #f6e58d;border-radius:10px;padding:8px 10px;font-size:13px;line-height:1.45}\n@media (max-width:980px){.grid{grid-template-columns:1fr} canvas{height:500px} .toolbar{grid-template-columns:1fr}}\n\n/* Single-screen workspace */\nhtml,body{width:100%;height:100%;overflow:hidden}\n.wrap{max-width:none;margin:0;padding:10px;height:100dvh}\n.grid{height:100%;grid-template-columns:minmax(0,1fr) minmax(270px,31%);gap:10px;align-items:stretch}\n.card{min-height:0;min-width:0;border-radius:12px}\n.grid>.card:first-child{display:flex;flex-direction:column}\n.head{padding:10px 14px 6px} h1{font-size:19px;margin:0 0 3px}.sub{font-size:11px;line-height:1.25}\n.toolbar{padding:8px 14px 5px;gap:8px;grid-template-columns:1fr 1fr}.toolbar label{font-size:10px;gap:3px}select{padding:6px 8px;font-size:13px;min-width:0}\n.modebar{padding:0 14px 5px;gap:5px}.pill{padding:4px 7px;font-size:11px}\n.canvasbox{flex:1;min-height:0;padding:4px 8px;display:flex}canvas{width:100%;height:100%;min-height:0;touch-action:none}\n.controls{padding:7px 12px 9px;gap:6px;grid-template-columns:1fr 1fr}.controls>.row{grid-column:1/-1;display:flex;flex-wrap:nowrap;gap:8px}.controls button{padding:7px 10px;font-size:12px}.range-row{grid-template-columns:65px minmax(0,1fr) auto;gap:5px}.range-row label,.small{font-size:10px}\n.side{height:100%;padding:9px;gap:7px;display:flex;flex-direction:column;justify-content:center}.statbox{padding:7px 9px;border-radius:9px}.statgrid{gap:6px 9px}.k{font-size:9px}.v{font-size:13px;margin-top:2px}.legend{gap:5px 8px}.leg,.note,.hl,.list li{font-size:11px;line-height:1.3}.list li{margin:0 0 3px}.hl{padding:5px 7px}.footer{font-size:10px}.side .statbox:nth-child(5){display:none}\n@media(max-height:650px){.sub,.footer,.side .statbox:nth-child(2){display:none}.side{gap:5px;padding:6px}.statbox{padding:5px 8px}.head{padding:7px 12px}.toolbar{padding:4px 12px}.modebar{padding-bottom:3px}.list li{font-size:10px}.side .statbox:nth-child(4) .list li:nth-child(n+4){display:none}}\n@media(max-width:760px){.wrap{padding:5px}.grid{grid-template-columns: minmax(0,1fr) minmax(185px,32%);gap:5px}.controls{grid-template-columns:1fr}.range-row{grid-template-columns:65px 1fr auto}.sub,.footer{display:none}.side{padding:5px}.statgrid{grid-template-columns:1fr 1fr}.v{font-size:11px}.head h1{font-size:15px}.toolbar{padding:5px}.modebar{padding:0 5px 3px}.statbox{padding:5px}.note,.hl,.list li{font-size:10px}}\n@media(max-width:520px) and (orientation:portrait){.grid{grid-template-columns:1fr;grid-template-rows:minmax(0,1fr) auto}.side{display:grid;grid-template-columns:1fr 1fr;gap:4px}.side .statbox:nth-child(2),.side .statbox:nth-child(3),.side .statbox:nth-child(4){display:none}.side .statbox:last-of-type{align-self:center}.controls{grid-template-columns:1fr 1fr}.range-row{grid-template-columns:1fr;gap:1px}.controls>.row{grid-column:1/-1}}\n\n\nbody{background:#e9eef4}.wrap{padding:8px}.grid{gap:9px;grid-template-columns:minmax(0,1fr) minmax(300px,29%)}.card{box-shadow:none;border-radius:10px}.head{padding:10px 15px}h1{font-size:clamp(18px,1.7vw,26px)}.sub{display:none}.toolbar{padding:8px 14px;gap:12px}.toolbar label{font-size:12px}select{font-size:16px;padding:8px}.modebar{display:none}.canvasbox{position:relative;padding:4px 10px}canvas{background:#fff;border:1px solid #d8e0e8;border-radius:8px}.controls{padding:8px 12px;gap:8px}.controls>.row button{width:auto;flex:1;font-size:14px;padding:9px}.range-row label,.small{font-size:13px}.range-row{grid-template-columns:80px minmax(0,1fr) auto}.side{justify-content:flex-start;gap:9px;padding:12px;height:100%;overflow:auto}.statbox{padding:12px;background:#f8fafc;border:1px solid #dce4ed}.k{font-size:clamp(11px,1vw,14px);color:#475569}.v{font-size:clamp(15px,1.35vw,23px);line-height:1.2}.statgrid{gap:12px}.statgrid>div:first-child{grid-column:1/-1}.note,.hl,.list li,.leg{font-size:clamp(13px,1.05vw,17px);line-height:1.4}.hl{background:transparent;border:0;padding:0}.list li{margin-bottom:7px}.side .statbox:nth-child(2){display:block}.side .statbox:nth-child(5){display:none}.footer{font-size:12px}.viewtools{position:absolute;top:15px;left:21px;right:21px;display:flex;gap:7px;align-items:center;flex-wrap:wrap;pointer-events:none}.viewtools>*{pointer-events:auto}.viewtools button,.viewtools select{width:auto;padding:7px 10px;font-size:13px;box-shadow:none;background:#fff;color:#17212b;border:1px solid #cbd5e1;border-radius:5px}.viewtools label{background:#fff;padding:6px;font-size:13px;font-weight:600}.viewtools .spacer{flex:1}.side #measure{font-size:clamp(15px,1.4vw,23px);font-weight:700}.side .statbox:last-of-type{flex:1;min-height:80px}.side .statbox:last-of-type .note{line-height:1.5}.viewhint{position:absolute;bottom:13px;left:24px;font-size:12px;color:#475569;pointer-events:none}\n@media(max-height:750px){.side{gap:6px;padding:8px}.statbox{padding:8px}.statgrid{gap:6px}.list li{margin-bottom:3px}.side .statbox:last-of-type{flex:0}.footer{display:none}.note,.hl,.list li,.leg{font-size:13px}.v{font-size:17px}.k{font-size:11px}}\n@media(max-width:760px){.grid{grid-template-columns:minmax(0,1fr) minmax(210px,35%)}.side{padding:5px}.viewtools{gap:3px}.viewtools button,.viewtools select,.viewtools label{font-size:10px;padding:4px}.controls{grid-template-columns:1fr}.toolbar select{font-size:12px}}\n</style>\n</head>\n<body>\n<div class=\"wrap\">\n  <div class=\"grid\">\n    <div class=\"card\">\n      <div class=\"head\">\n        <h1>PANALDO · 3D Normal Mode Viewer</h1>\n        <div class=\"sub\">Four protonation states · 225 ORCA normal modes · Drag to rotate</div>\n      </div>\n      <div class=\"toolbar\">\n        <label>Species\n          <select id=\"species\"></select>\n        </label>\n        <label>Mode\n          <select id=\"mode\"></select>\n        </label>\n      </div>\n      <div class=\"modebar\">\n        <div class=\"pill\" id=\"pillSpecies\">Species</div>\n        <div class=\"pill\" id=\"pillMode\">Mode</div>\n        <div class=\"pill\" id=\"pillFreq\">Frequency</div>\n        <div class=\"pill\" id=\"pillIR\">IR intensity</div>\n      </div>\n      <div class=\"canvasbox\"><canvas id=\"cv\" width=\"860\" height=\"620\"></canvas><div class=\"viewtools\"><select id=\"representation\" aria-label=\"Molecular representation\"><option value=\"ball\">Ball &amp; stick</option><option value=\"stick\">Stick</option><option value=\"cpk\">Space filling</option></select><label><input id=\"hydrogen\" type=\"checkbox\" checked> H atoms</label><label><input id=\"labels\" type=\"checkbox\" checked> Labels</label><label><input id=\"highlight\" type=\"checkbox\" checked> Highlight</label><span class=\"spacer\"></span><button id=\"zoomOut\" title=\"Zoom out\">−</button><button id=\"zoomIn\" title=\"Zoom in\">+</button><button id=\"fullscreen\">Full screen</button><button id=\"export\">PNG</button></div><div class=\"viewhint\">Drag to rotate · Mouse wheel to zoom</div></div>\n      <div class=\"controls\">\n        <div class=\"row\">\n          <button id=\"playBtn\">Pause animation</button>\n          <button id=\"resetBtn\" class=\"secondary\">Reset view</button>\n        </div>\n        <div class=\"range-row\">\n          <label for=\"amp\">Amplitude</label>\n          <input id=\"amp\" type=\"range\" min=\"0.05\" max=\"0.30\" step=\"0.01\" value=\"0.24\">\n          <span id=\"ampVal\" class=\"small\">0.24 Å</span>\n        </div>\n        <div class=\"range-row\">\n          <label for=\"speed\">Speed</label>\n          <input id=\"speed\" type=\"range\" min=\"0.01\" max=\"0.12\" step=\"0.005\" value=\"0.05\">\n          <span id=\"speedVal\" class=\"small\">0.050 rad/frame</span>\n        </div>\n      </div>\n    </div>\n\n    <div class=\"card\">\n      <div class=\"side\">\n        <div class=\"statbox\">\n          <div class=\"statgrid\">\n            <div><div class=\"k\">Selected species</div><div class=\"v\" id=\"statSpecies\">—</div></div>\n            <div><div class=\"k\">Selected mode</div><div class=\"v\" id=\"statMode\">—</div></div>\n            <div><div class=\"k\">Frequency</div><div class=\"v\" id=\"statFreq\">—</div></div>\n            <div><div class=\"k\">IR intensity</div><div class=\"v\" id=\"statIR\">—</div></div>\n            <div><div class=\"k\">Atoms</div><div class=\"v\" id=\"statAtoms\">—</div></div>\n            <div><div class=\"k\">Positive modes</div><div class=\"v\" id=\"statPosModes\">—</div></div>\n          </div>\n        </div>\n\n        <div class=\"statbox\">\n          <div class=\"k\" style=\"margin-bottom:10px\">Atom colours</div>\n          <div class=\"legend\">\n            <div class=\"leg\"><span class=\"dot\" style=\"background:#4b5563\"></span>Carbon</div>\n            <div class=\"leg\"><span class=\"dot\" style=\"background:#2563eb\"></span>Nitrogen</div>\n            <div class=\"leg\"><span class=\"dot\" style=\"background:#ef4444\"></span>Oxygen</div>\n            <div class=\"leg\"><span class=\"dot\" style=\"background:#f9fafb;border:1px solid #d1d5db\"></span>Hydrogen</div>\n          </div>\n        </div>\n\n        <div class=\"statbox\">\n          <div class=\"k\" style=\"margin-bottom:8px\">What to watch</div>\n          <div class=\"hl\" id=\"focusBox\">Choose a species and mode. The viewer will highlight the dominant bond-motion region from the selected normal mode.</div>\n        </div>\n\n        <div class=\"statbox\">\n          <div class=\"k\" style=\"margin-bottom:8px\">Top bond changes in this mode</div>\n          <ol id=\"topBonds\" class=\"list\"></ol>\n        </div>\n\n        <div class=\"statbox\">\n          <div class=\"k\" style=\"margin-bottom:8px\">How to use</div>\n          <div class=\"note\">Drag to rotate the structure. Use <b>Amplitude</b> to exaggerate motion for teaching. Use <b>Mode</b> to move across all positive-frequency vibrational modes. The motion is visually amplified, but the pattern comes from the real ORCA displacement vectors.</div>\n        </div>\n\n        <div class=\"statbox\">\n          <div class=\"k\" style=\"margin-bottom:8px\">Live measurement</div>\n          <div id=\"measure\" class=\"note\">Highlighted bond distance: -- Å</div>\n        </div>\n\n        <div class=\"footer\">Standalone HTML · no internet required · polished all-four-molecule version.</div>\n      </div>\n    </div>\n  </div>\n</div>\n\n<script>\nconst DATA = __PANALDO_DATA__;\nconst SPECIES_ORDER = [\"P01_neutral\", \"P02_carbonyl_O\", \"P03_amide_N\", \"P04_phenolic_O\"];\nconst COLORS = { C:\"#4b5563\", N:\"#2563eb\", O:\"#ef4444\", H:\"#f9fafb\" };\nconst EDGES = { C:\"#374151\", N:\"#1d4ed8\", O:\"#b91c1c\", H:\"#d1d5db\" };\nconst RADII = { C:17, N:18, O:18, H:10 };\n\nconst speciesSel = document.getElementById('species');\nconst modeSel = document.getElementById('mode');\nconst canvas = document.getElementById('cv');\nconst ctx = canvas.getContext('2d');\nlet rotX = 0.38, rotY = -0.52, phase = 0, playing = true, dragging = false, lastX = 0, lastY = 0;\nlet amplitude = 0.12, speed = 0.05, zoom=1, W=860,H=620, pixelRatio=1, baseScale=70;\nconst $=id=>document.getElementById(id);\n$(\"amp\").value=amplitude;\n\nfor (const key of SPECIES_ORDER) {\n  const opt = document.createElement('option');\n  opt.value = key; opt.textContent = DATA[key].label; speciesSel.appendChild(opt);\n}\n\nfunction currentSpecies() { return DATA[speciesSel.value]; }\nfunction currentMode() {\n  const sp = currentSpecies();\n  return sp.modes.find(m => m.mode === Number(modeSel.value));\n}\n\nfunction loadModes() {\n  const sp = currentSpecies();\n  modeSel.innerHTML = '';\n  for (const m of sp.modes) {\n    const opt = document.createElement('option');\n    opt.value = m.mode;\n    opt.textContent = `${m.mode} | ${m.frequency.toFixed(2)} cm⁻¹`;\n    modeSel.appendChild(opt);\n  }\n  modeSel.value = sp.default_mode;\n  updateInfo();\n}\n\nfunction updateInfo() {\n  const sp = currentSpecies();\n  const mode = currentMode();\n  document.getElementById('pillSpecies').textContent = sp.label;\n  document.getElementById('pillMode').textContent = `Mode ${mode.mode}`;\n  document.getElementById('pillFreq').textContent = `${mode.frequency.toFixed(2)} cm⁻¹`;\n  document.getElementById('pillIR').textContent = `IR ${mode.intensity.toFixed(2)} km/mol`;\n  document.getElementById('statSpecies').textContent = sp.label;\n  document.getElementById('statMode').textContent = mode.mode;\n  document.getElementById('statFreq').textContent = `${mode.frequency.toFixed(2)} cm⁻¹`;\n  document.getElementById('statIR').textContent = `${mode.intensity.toFixed(2)} km/mol`;\n  document.getElementById('statAtoms').textContent = sp.symbols.length;\n  document.getElementById('statPosModes').textContent = sp.modes.length;\n\n  const focus = mode.top_bonds[0];\n  document.getElementById('focusBox').innerHTML = `<b>Main highlighted bond:</b> ${focus.bond}. Largest projected bond-length displacement; not a formal vibrational assignment.`;\n  const ol = document.getElementById('topBonds');\n  ol.innerHTML = '';\n  for (const b of mode.top_bonds) {\n    const li = document.createElement('li');\n    li.innerHTML = `<b>${b.bond}</b> · relative stretch = ${b.abs.toFixed(3)}`;\n    ol.appendChild(li);\n  }\n  document.getElementById('ampVal').textContent = amplitude.toFixed(2) + ' Å';\n  document.getElementById('speedVal').textContent = speed.toFixed(3) + ' rad/frame';\n}\n\nfunction rotated(pt) {\n  let [x,y,z] = pt;\n  const x1 = x*Math.cos(rotY) + z*Math.sin(rotY);\n  const z1 = -x*Math.sin(rotY) + z*Math.cos(rotY);\n  const y1 = y*Math.cos(rotX) - z1*Math.sin(rotX);\n  const z2 = y*Math.sin(rotX) + z1*Math.cos(rotX);\n  return [x1,y1,z2];\n}\n\nfunction currentAtoms() {\n  const sp = currentSpecies();\n  const mode = currentMode();\n  const maxD=Math.max(...mode.d.map(v=>Math.hypot(...v)));\n  const t = Math.sin(phase) * amplitude / (maxD || 1);\n  return sp.xyz.map((p, i) => [\n    p[0] + mode.d[i][0] * t,\n    p[1] + mode.d[i][1] * t,\n    p[2] + mode.d[i][2] * t,\n  ]);\n}\n\nfunction project(atom) {\n  const [x,y,z] = rotated(atom);\n  const scale = baseScale*zoom;\n  return { x: W/2 + x*scale, y: H/2 - y*scale + 13, z, scale };\n}\n\nfunction drawBond(a,b,ra,rb,highlight=false) {\n  const dx = b.x-a.x, dy = b.y-a.y, L = Math.hypot(dx,dy)||1;\n  const ux = dx/L, uy = dy/L;\n  const x1 = a.x + ux*ra*0.9, y1 = a.y + uy*ra*0.9;\n  const x2 = b.x - ux*rb*0.9, y2 = b.y - uy*rb*0.9;\n  ctx.strokeStyle = highlight ? '#f59e0b' : '#9ca3af';\n  ctx.lineWidth = highlight ? 10 : 7;\n  ctx.lineCap = 'round';\n  ctx.beginPath(); ctx.moveTo(x1,y1); ctx.lineTo(x2,y2); ctx.stroke();\n  ctx.strokeStyle = highlight ? '#fde68a' : '#dfe5ec';\n  ctx.lineWidth = highlight ? 4 : 3;\n  ctx.beginPath(); ctx.moveTo(x1,y1); ctx.lineTo(x2,y2); ctx.stroke();\n}\n\nfunction drawAtom(sym,p) {\n  const r = radius(sym);\n  const grad = ctx.createRadialGradient(p.x-r*0.35,p.y-r*0.45,1,p.x,p.y,r);\n  grad.addColorStop(0,'#ffffff');\n  grad.addColorStop(0.35,COLORS[sym]);\n  grad.addColorStop(1,EDGES[sym]);\n  ctx.beginPath(); ctx.arc(p.x,p.y,r,0,Math.PI*2);\n  ctx.fillStyle = grad;\n  ctx.shadowColor = 'rgba(15,23,42,0.18)'; ctx.shadowBlur = 3; ctx.shadowOffsetY = 1;\n  ctx.fill();\n  ctx.shadowBlur = 0; ctx.shadowOffsetY = 0;\n  ctx.strokeStyle = sym==='H' ? '#cfd5dc' : 'rgba(0,0,0,0.16)';\n  ctx.lineWidth = 1; ctx.stroke();\n  if ($('labels').checked && sym !== 'H') {\n    ctx.fillStyle = 'white';\n    ctx.font = `700 ${Math.max(12,Math.min(23,r*.85))}px Segoe UI, Arial`; ctx.textAlign='center'; ctx.textBaseline='middle';\n    ctx.fillText(sym, p.x, p.y+0.5);\n  }\n}\n\nfunction radius(sym){const rep=$('representation').value; return baseScale*zoom*(rep==='cpk'?({C:1.7,N:1.55,O:1.52,H:1.2}[sym]):rep==='stick'?.085:({C:.24,N:.25,O:.25,H:.145}[sym]));}\nfunction draw() {\n  ctx.setTransform(pixelRatio,0,0,pixelRatio,0,0);ctx.fillStyle='#fff';ctx.fillRect(0,0,W,H);\n  const sp = currentSpecies();\n  const mode = currentMode();\n  const atoms = currentAtoms();\n  const eq=sp.xyz.map(rotated); const maxX=Math.max(...eq.map(p=>Math.abs(p[0])))+.75; const maxY=Math.max(...eq.map(p=>Math.abs(p[1])))+.75; baseScale=Math.min(W*.45/maxX,(H-90)*.46/maxY); const proj = atoms.map(project); const visible=i=>$('hydrogen').checked||sp.symbols[i]!=='H';\n  const hi = mode.top_bonds[0].atoms;\n  const hiKey = new Set([`${hi[0]}-${hi[1]}`, `${hi[1]}-${hi[0]}`]);\n\n  const objects = [];\n  for (const [i,j] of sp.bonds) if(visible(i)&&visible(j)&&$('representation').value!=='cpk') objects.push({type:'bond', i, j, z:(proj[i].z+proj[j].z)/2});\n  proj.forEach((p,i)=>{if(visible(i))objects.push({type:'atom', i, z:p.z})});\n  objects.sort((a,b)=>a.z-b.z);\n\n  for (const obj of objects) {\n    if (obj.type === 'bond') {\n      const a = proj[obj.i], b = proj[obj.j];\n      const highlight = $('highlight').checked && hiKey.has(`${obj.i}-${obj.j}`);\n      drawBond(a,b,radius(sp.symbols[obj.i]),radius(sp.symbols[obj.j]),highlight);\n    } else {\n      drawAtom(sp.symbols[obj.i], proj[obj.i]);\n    }\n  }\n\n  // highlight top bond atoms\n  for (const idx of hi) { if(!$('highlight').checked||!visible(idx))continue;\n    const p = proj[idx];\n    ctx.strokeStyle = 'rgba(245,158,11,0.60)';\n    ctx.lineWidth = 2;\n    ctx.setLineDash([6,4]);\n    ctx.beginPath(); ctx.arc(p.x,p.y,radius(sp.symbols[idx])+6,0,Math.PI*2); ctx.stroke();\n    ctx.setLineDash([]);\n  }\n\n  const dx = atoms[hi[0]][0]-atoms[hi[1]][0], dy = atoms[hi[0]][1]-atoms[hi[1]][1], dz = atoms[hi[0]][2]-atoms[hi[1]][2];\n  const dist = Math.sqrt(dx*dx+dy*dy+dz*dz);\n  document.getElementById('measure').textContent = `${mode.top_bonds[0].bond}: ${dist.toFixed(3)} Å (animated)`;\n}\n\nfunction fitCanvas(){\n const r=canvas.getBoundingClientRect();\n W=Math.max(1,r.width);H=Math.max(1,r.height);pixelRatio=Math.min(3,window.devicePixelRatio||1);canvas.width=Math.round(W*pixelRatio);canvas.height=Math.round(H*pixelRatio);\n draw();\n}\nnew ResizeObserver(fitCanvas).observe(canvas);\nfunction tick() { if (playing) phase += speed; draw(); requestAnimationFrame(tick); }\n\nspeciesSel.addEventListener('change', ()=>{ phase=0; loadModes(); draw(); });\nmodeSel.addEventListener('change', ()=>{ phase=0; updateInfo(); draw(); });\ndocument.getElementById('playBtn').onclick = ()=>{ playing=!playing; document.getElementById('playBtn').textContent = playing ? 'Pause animation' : 'Play animation'; };\ndocument.getElementById('resetBtn').onclick = ()=>{ rotX=0.38; rotY=-0.52; phase=0; zoom=1; draw(); };\ndocument.getElementById('amp').oninput = e => { amplitude=Number(e.target.value); document.getElementById('ampVal').textContent = amplitude.toFixed(2)+' Å'; if(!playing) draw(); };\ndocument.getElementById('speed').oninput = e => { speed=Number(e.target.value); document.getElementById('speedVal').textContent = speed.toFixed(3)+' rad/frame'; };\n\ncanvas.onpointerdown = e => { dragging=true; lastX=e.clientX; lastY=e.clientY; canvas.setPointerCapture(e.pointerId); };\ncanvas.onpointermove = e => { if(!dragging) return; rotY += (e.clientX-lastX)*0.012; rotX += (e.clientY-lastY)*0.012; lastX=e.clientX; lastY=e.clientY; draw(); };\ncanvas.onpointerup = ()=> dragging=false;\ncanvas.onpointercancel = ()=> dragging=false;\n\nfunction changeZoom(f){zoom=Math.max(.45,Math.min(2.5,zoom*f));draw()};$('zoomIn').onclick=()=>changeZoom(1.12);$('zoomOut').onclick=()=>changeZoom(1/1.12);canvas.addEventListener('wheel',e=>{e.preventDefault();changeZoom(e.deltaY<0?1.08:1/1.08)},{passive:false});for(const id of ['representation','hydrogen','labels','highlight'])$(id).onchange=draw;$('fullscreen').onclick=()=>{if(document.fullscreenElement)document.exitFullscreen();else document.documentElement.requestFullscreen()};$('export').onclick=()=>{const a=document.createElement('a');a.download=`PANALDO_${speciesSel.value}_mode${modeSel.value}.png`;a.href=canvas.toDataURL('image/png');a.click()};\nspeciesSel.value = SPECIES_ORDER[0];\nloadModes();\nfitCanvas();\nrequestAnimationFrame(tick);\n</script>\n</body>\n</html>"
html = template.replace('__PANALDO_DATA__', json.dumps(viewer_data, ensure_ascii=False))
output = Path('PANALDO_vibration_viewer_v3.html')
output.write_text(html, encoding='utf-8')
display(HTML('<a href="PANALDO_vibration_viewer_v3.html" target="_blank">Open PANALDO v3 viewer</a>'))
